<a href="https://colab.research.google.com/github/SunilSharmaNP/VidLM/blob/main/ssleech_hk_deploy.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

<h1 align='center'>🤖 SSLeech — Heroku Deployer</h1>

<center><img src='https://te.legra.ph/file/8086f391e542ed5c6a4c2.jpg' height='200' width='400' alt='SSLeech'/></center>

---

### ***Collab Repository Details***
- 🔗 **Bot Repo :** https://github.com/SunilSharmaNP/SSLeech/tree/ssleech-hk
- 🚀 **Deploy Repo :** https://github.com/SunilSharmaNP/VidLM
- ☢️ **Colab Version :** _v2.0_

---
### ***Deploy SSLeech in Heroku using Google Colab***

In [ ]:
#@title <center><h3>***Heroku Login***</h3></center><br>

#@markdown ---

Heroku_Email = "" #@param {type:"string"}
Heroku_API   = "" #@param {type:"string"}
#@markdown <h6>( <b>Note:</b> <i>Get API key: https://dashboard.heroku.com/account → API Key</i> )</h6>

#@markdown ---

!curl -s https://cli-assets.heroku.com/install.sh | sh

from IPython.display import HTML, clear_output, display
clear_output()
display(HTML("<marquee><b>Heroku CLI Installed !</b></marquee>"))

if not all([Heroku_Email, Heroku_API]):
    raise ValueError("Please fill in Heroku_Email and Heroku_API before running!")

from os import path as ospath, chmod

netrc_path = ospath.expanduser("~/.netrc")
netrc_creds = f'''machine api.heroku.com
  login {Heroku_Email}
  password {Heroku_API}
machine git.heroku.com
  login {Heroku_Email}
  password {Heroku_API}'''

with open(netrc_path, "w") as netrc_file:
    netrc_file.write(netrc_creds)
chmod(netrc_path, 0o600)

!git config --global user.email {Heroku_Email}
!git config --global user.name "SSLeech"

display(HTML("<marquee><b>✅ Heroku Email & API Loaded!</b></marquee>"))

In [ ]:
#@title <center><h3>***Create Heroku Multi App***</h3></center><br>

#@markdown ---

App_Names     = "" #@param {type:"string"}
#@markdown <h6>( <b>Syntax:</b> <i>bot_name1 bot_name2, separated by space !</i> )</h6>
#@markdown <h6>( <b>Note:</b> <i>App Name is Optional, skip for random name !</i> )</h6>

Server_Region = "us" #@param ["us", "eu"] {allow-input: true}
HK_Team_Name  = "" #@param {type:"string"}
#@markdown <h6>( <b>Note:</b> <i>Optional — only if deploying to a Heroku Team</i> )</h6>

#@markdown ---

HK_Team_Name = f"--team {HK_Team_Name}" if HK_Team_Name else ""
for App_Name in (App_Names.split() or [""]):
    !heroku create --region $Server_Region --stack container $HK_Team_Name $App_Name

In [ ]:
#@title <center><h3>***SSLeech Config Setup***</h3></center><br>

#@markdown ---
App_Name = "" #@param {type:"string"}
#@markdown <h6>( <b>Note:</b> <i>Config Setup for this App Name. Change App Name for every Config Save!</i> )</h6>

#@markdown ---
#@markdown #### ***Fill all Mandatory Variables*** **(All Required)**

BOT_TOKEN      = "" #@param {type:"string"}
TELEGRAM_API   = 0  #@param {type:"integer"}
TELEGRAM_HASH  = "" #@param {type:"string"}
OWNER_ID       = 0  #@param {type:"integer"}
DATABASE_URL   = "" #@param {type:"string"}
BASE_URL       = "" #@param {type:"string"}
#@markdown <h6>( <i>BASE_URL example: https://your-app.herokuapp.com</i> )</h6>

#@markdown ---
#@markdown #### ***Optional Variables***

UPSTREAM_REPO   = "https://github.com/SunilSharmaNP/SSLeech" #@param {type:"string"}
UPSTREAM_BRANCH = "ssleech-hk" #@param {type:"string"}
TIMEZONE        = "Asia/Kolkata" #@param {type:"string"}

#@markdown ---

if not App_Name:
    raise ValueError("❌ App_Name is required!")
if not all([BOT_TOKEN, TELEGRAM_API, TELEGRAM_HASH, OWNER_ID, DATABASE_URL, BASE_URL]):
    raise ValueError("❌ Please fill in all Mandatory Variables.")

from os import path, remove
import sys, subprocess
from IPython.display import HTML, display

# ── Clone deploy repo ────────────────────────────────────────────────────────
if path.isdir(App_Name):
    !rm -rf $App_Name

!git clone https://github.com/SunilSharmaNP/VidLM $App_Name
%cd $App_Name

for f in ["README.md", "ssleech_hk_deploy.ipynb"]:
    if path.isfile(f):
        remove(f)

# ── Save ALL vars to MongoDB ──────────────────────────────────────────────────
# update.py reads from MongoDB on every Heroku restart — no need to store
# TELEGRAM_API, OWNER_ID etc. in Heroku Config Vars after first deploy.
subprocess.run([sys.executable, "-m", "pip", "install", "pymongo[srv]", "-q"])
from pymongo.mongo_client import MongoClient
from pymongo.server_api import ServerApi

BOT_ID = BOT_TOKEN.split(":", 1)[0]
config_doc = {
    "_id":             BOT_ID,
    "BOT_TOKEN":       BOT_TOKEN,
    "TELEGRAM_API":    str(TELEGRAM_API),
    "TELEGRAM_HASH":   TELEGRAM_HASH,
    "OWNER_ID":        str(OWNER_ID),
    "DATABASE_URL":    DATABASE_URL,
    "BASE_URL":        BASE_URL,
    "UPSTREAM_REPO":   UPSTREAM_REPO,
    "UPSTREAM_BRANCH": UPSTREAM_BRANCH,
    "TIMEZONE":        TIMEZONE,
}
try:
    conn = MongoClient(DATABASE_URL, server_api=ServerApi("1"), serverSelectionTimeoutMS=10000)
    conn.wzmlx.settings.config.replace_one({"_id": BOT_ID}, config_doc, upsert=True)
    conn.close()
    display(HTML("<b style='color:green'>✅ Config saved to MongoDB!</b>"))
except Exception as e:
    display(HTML(f"<b style='color:red'>❌ MongoDB error: {e}</b>"))
    raise

# ── Write config.env for first Heroku build ───────────────────────────────────
with open("config.env", "w") as f:
    for k, v in config_doc.items():
        if k == "_id":
            continue
        f.write(f'{k}="{v}"\n')
display(HTML("<b style='color:green'>✅ config.env written!</b>"))

# ── Set BOT_TOKEN + DATABASE_URL in Heroku Config Vars ───────────────────────
!heroku config:set BOT_TOKEN=$BOT_TOKEN DATABASE_URL=$DATABASE_URL --app $App_Name

%cd ..

print("\n📦 All Available Config Bot Names:")
!ls

In [ ]:
# @title <center><h3>***Deploy Heroku Multi App***</h3></center><br>

#@markdown ---
App_Names = "" #@param {type:"string"}
#@markdown <h6>( <b>Syntax:</b> <i>bot_name1 bot_name2, separated by space ! Config must be set for each app first</i> )</h6>

#@markdown ---

from os import path as ospath
from IPython.display import HTML, display

for App_Name in App_Names.split():
    if ospath.isdir(App_Name):
        display(HTML(f"<hr><b>🚀 Deploying: {App_Name}</b>"))
        %cd $App_Name
        !git add . -f
        !git commit -m "SSLeech Deploy"
        !heroku git:remote -a $App_Name
        !git push heroku main -f
        %cd ..
        display(HTML(f"<b style='color:green'>✅ {App_Name} deployed! → https://{App_Name}.herokuapp.com</b>"))
    else:
        display(HTML(f"<b style='color:red'>❌ Config not set for <code>{App_Name}</code> — run the Config cell first!</b>"))

In [ ]:
# @title <center><h3>***Show Heroku App Logs***</h3></center><br>

#@markdown ---
App_Name = "" #@param {type:"string"}
#@markdown ---

!heroku logs -t -a {App_Name}

In [ ]:
#@title <center><h3>***Restart App***</h3></center><br>

#@markdown ---
App_Name = "" #@param {type:"string"}
#@markdown ---

!heroku restart -a {App_Name}

from IPython.display import HTML, display
display(HTML(f"<b style='color:green'>✅ {App_Name} restarting...</b>"))

In [ ]:
#@title <center><h3>***Heroku Logout***</h3></center><br>

!heroku logout